In [2]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

In [3]:
pdf_path = "./Docs/Designing Instruction with Generative AI.pdf"

loader = PyPDFLoader(pdf_path)

pdf_docs = loader.load()

pdf_docs

Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 14.0 (Macintosh)', 'creationdate': '2025-07-17T11:36:07+05:30', 'author': 'Brent A. Anders', 'keywords': 'lifelong learning; higher education; technology in education; open education; computers and technology; adult education; eLearning; distance education; classroom practice', 'moddate': '2025-10-14T17:13:09+05:30', 'subject': 'Designing Instruction with Generative AI offers a novel set of tools and strategies for leveraging generative AI to create engaging and personalized learning experiences. While instructional designers are a tremendous asset to higher education, not all colleges or universities have the robust staff needed to support all instructors on staff or large student populations. Drawing on a wealth of research, professional experience, and strategic insights, this book equips new and seasoned teaching faculty and trainers with step-by-step directions on how freely accessible artificia

In [4]:
# for doc in pdf_docs:        ##for adding metadata
#     doc.metadata["documentID"] = "DOC1"
#     doc.metadata["source"] = "ABC"

In [5]:
for doc in pdf_docs:

    doc.metadata = {"source": "GenAI_pdf",
                    "documentID": "DOC1"}

pdf_docs

[Document(metadata={'source': 'GenAI_pdf', 'documentID': 'DOC1'}, page_content=''),
 Document(metadata={'source': 'GenAI_pdf', 'documentID': 'DOC1'}, page_content='Designing Instruction \nwith Generative AI \nDesigning Instruction with Generative AI offers a novel set of  tools and strategies \nfor leveraging generative AI to create engaging and personalized learning \nexperiences. While instructional designers are a tremendous asset to higher \neducation, not all colleges or universities have the robust staff  needed to sup-\nport all instructors on staff  or large student populations. Drawing on a wealth \nof  research, professional experience, and strategic insights, this book equips \nnew and seasoned teaching faculty and trainers with step-by-step directions \non how freely accessible artificial intelligence software can assist with all \naspects of  the course creation and instruction process and cater to the needs \nof  diverse learners. Each chapter offers forward-thinking and 

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
# Split documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

split_docs = splitter.split_documents(pdf_docs)

In [8]:
import hashlib

In [9]:
# Create chunks + SHA256 IDs
# -----------------------------
gen_chunks = []
ids = []

for chunk in split_docs:

    # Copy metadata
    metadata = chunk.metadata.copy()

    # Generate SHA256 hash ID
    hash_id = hashlib.sha256(
        chunk.page_content.encode()
    ).hexdigest()

    metadata["chunk_id"] = hash_id

    gen_chunks.append(
        Document(
            page_content=chunk.page_content,
            metadata=metadata
        )
    )

    ids.append(hash_id)

In [10]:
len(gen_chunks)

503

In [11]:
gen_chunks[0].metadata

{'source': 'GenAI_pdf',
 'documentID': 'DOC1',
 'chunk_id': 'e29d9de8f836d572ba84f5ab95434e68601d0563644cf0b702c2395a9f00065e'}

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

c:\Users\User\Desktop\Projects\GenAI_Projects\AI_PDF_Question_Answering_App\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# Embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 693.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
from chromadb import PersistentClient

client = PersistentClient(path="chroma_db")

print(client.list_collections())


[Collection(name=genai_docs), Collection(name=ai_docs)]


In [ ]:
# ###delete the collection
#
# client.delete_collection("genai_docs")
#
# print(client.list_collections())

In [17]:
from langchain_community.vectorstores import Chroma

In [18]:
# Load vector DB
vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embedding_model,
    collection_name="genai_docs"
)

C:\Users\User\AppData\Local\Temp\ipykernel_8508\22090142.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [19]:
# Add documents (prevents duplicates)
vectorstore.add_documents(
    documents=gen_chunks,
    ids=ids
)

['...']

In [20]:
vectorstore._collection.count()

503